# NMF on ModulAir 00691

In [7]:
from sklearn.decomposition import NMF
import numpy as np
import pandas as pd

## Cleaning Data

In [8]:
#importing data from Modulair MOD-00691
data = pd.read_csv('MOD-00691.csv').set_index('timestamp')
data.head()

,id,timestamp_local,sn,rh,temp,bin0,bin1,bin2,bin3,bin4,...,no2,o3,pm1_model_id,pm25_model_id,pm10_model_id,co_model_id,no_model_id,no2_model_id,o3_model_id,ws_scalar
timestamp,,,,,,,,,,,,,,,,,,,,,
2025-12-31T23:59:34Z,577611104,2025-12-31T18:59:34Z,MOD-00691,51.5,0.0,6.169,0.652,0.217,0.057,0.027,...,37.685,24.425,14346,14347,14348,14478.0,14503.0,14553.0,14528.0,1.35
2025-12-31T23:58:34Z,577611106,2025-12-31T18:58:34Z,MOD-00691,51.4,0.0,6.513,0.912,0.194,0.036,0.022,...,36.979,24.792,14346,14347,14348,14478.0,14503.0,14553.0,14528.0,0.86
2025-12-31T23:57:34Z,577611105,2025-12-31T18:57:34Z,MOD-00691,51.2,0.0,6.793,0.800,0.232,0.062,0.022,...,36.276,25.170,14346,14347,14348,14478.0,14503.0,14553.0,14528.0,2.51
2025-12-31T23:56:34Z,577611107,2025-12-31T18:56:34Z,MOD-00691,51.8,0.0,6.516,0.755,0.209,0.047,0.047,...,37.199,24.746,14346,14347,14348,14478.0,14503.0,14553.0,14528.0,1.39
2025-12-31T23:55:34Z,577609047,2025-12-31T18:55:34Z,MOD-00691,51.6,-0.1,6.421,0.860,0.280,0.026,0.036,...,36.963,25.476,14346,14347,14348,14478.0,14503.0,14553.0,14528.0,1.39


In [9]:
#only columns of interest
COLS_TO_INCLUDE = ['timestamp_local','co','no','no2','o3','bin0','bin1','bin2','bin3','bin4','bin5']
data = data[COLS_TO_INCLUDE]

data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
timestamp,,,,,,,,,,,
2025-12-31T23:59:34Z,2025-12-31T18:59:34Z,1158.375,2.424,37.685,24.425,6.169,0.652,0.217,0.057,0.027,0.004
2025-12-31T23:58:34Z,2025-12-31T18:58:34Z,1285.293,2.293,36.979,24.792,6.513,0.912,0.194,0.036,0.022,0.018
2025-12-31T23:57:34Z,2025-12-31T18:57:34Z,1714.006,2.812,36.276,25.170,6.793,0.800,0.232,0.062,0.022,0.018
2025-12-31T23:56:34Z,2025-12-31T18:56:34Z,2682.790,2.424,37.199,24.746,6.516,0.755,0.209,0.047,0.047,0.017
2025-12-31T23:55:34Z,2025-12-31T18:55:34Z,1489.066,2.424,36.963,25.476,6.421,0.860,0.280,0.026,0.036,0.026


In [10]:
#removing the UTC time
data = data.reset_index(drop = True)
data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-12-31T18:59:34Z,1158.375,2.424,37.685,24.425,6.169,0.652,0.217,0.057,0.027,0.004
1,2025-12-31T18:58:34Z,1285.293,2.293,36.979,24.792,6.513,0.912,0.194,0.036,0.022,0.018
2,2025-12-31T18:57:34Z,1714.006,2.812,36.276,25.170,6.793,0.800,0.232,0.062,0.022,0.018
3,2025-12-31T18:56:34Z,2682.790,2.424,37.199,24.746,6.516,0.755,0.209,0.047,0.047,0.017
4,2025-12-31T18:55:34Z,1489.066,2.424,36.963,25.476,6.421,0.860,0.280,0.026,0.036,0.026


In [11]:
#converting to datetime
data['timestamp_local'] = pd.to_datetime(data['timestamp_local'],
                                       format='%Y-%m-%dT%H:%M:%SZ',
                                       exact=False)
data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-12-31 18:59:34,1158.375,2.424,37.685,24.425,6.169,0.652,0.217,0.057,0.027,0.004
1,2025-12-31 18:58:34,1285.293,2.293,36.979,24.792,6.513,0.912,0.194,0.036,0.022,0.018
2,2025-12-31 18:57:34,1714.006,2.812,36.276,25.170,6.793,0.800,0.232,0.062,0.022,0.018
3,2025-12-31 18:56:34,2682.790,2.424,37.199,24.746,6.516,0.755,0.209,0.047,0.047,0.017
4,2025-12-31 18:55:34,1489.066,2.424,36.963,25.476,6.421,0.860,0.280,0.026,0.036,0.026


In [12]:
#taking hourly average of df. round to floor of the hour
data = data.groupby(data['timestamp_local'].dt.floor('h')).agg(co = ('co','mean'),
                                                         no2 = ('no2','mean'),
                                                         o3 = ('o3','mean'),
                                                         no = ('no','mean'),
                                                         bin0 = ('bin0','mean'),
                                                         bin1 = ('bin1','mean'),
                                                         bin2 = ('bin2','mean'),
                                                         bin3 = ('bin3','mean'),
                                                         bin4 = ('bin4','mean'),
                                                         bin5 = ('bin5','mean')).reset_index()

data = data.round(decimals = 2)
data = data.dropna()
data

,timestamp_local,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-11-22 21:00:00,773.24,34.55,28.07,2.14,6.58,0.71,0.18,0.03,0.02,0.01
1,2025-11-22 22:00:00,742.28,33.38,29.32,2.12,3.41,0.52,0.18,0.04,0.02,0.01
2,2025-11-22 23:00:00,720.22,32.79,30.59,2.07,2.72,0.36,0.12,0.02,0.02,0.01
3,2025-11-23 00:00:00,765.60,34.45,27.34,2.17,2.81,0.34,0.10,0.01,0.01,0.01
4,2025-11-23 01:00:00,869.31,35.49,24.78,2.40,3.37,0.39,0.12,0.02,0.02,0.01
...,...,...,...,...,...,...,...,...,...,...,...
903,2025-12-31 14:00:00,1195.91,35.29,32.69,2.04,6.81,0.61,0.18,0.03,0.03,0.02
904,2025-12-31 15:00:00,1023.29,35.38,32.82,1.99,6.27,0.61,0.17,0.03,0.03,0.02
905,2025-12-31 16:00:00,1021.94,36.26,30.29,2.22,7.37,0.80,0.22,0.04,0.03,0.02
906,2025-12-31 17:00:00,946.88,37.36,26.09,2.02,7.94,0.93,0.26,0.05,0.04,0.02


In [13]:
#setting local time as index
data = data.set_index('timestamp_local')
data.head()

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-11-22 21:00:00,773.24,34.55,28.07,2.14,6.58,0.71,0.18,0.03,0.02,0.01
2025-11-22 22:00:00,742.28,33.38,29.32,2.12,3.41,0.52,0.18,0.04,0.02,0.01
2025-11-22 23:00:00,720.22,32.79,30.59,2.07,2.72,0.36,0.12,0.02,0.02,0.01
2025-11-23 00:00:00,765.60,34.45,27.34,2.17,2.81,0.34,0.10,0.01,0.01,0.01
2025-11-23 01:00:00,869.31,35.49,24.78,2.40,3.37,0.39,0.12,0.02,0.02,0.01


In [14]:
#subsetting for only positive and non NA values
data = data.where(data>0)
data = data.dropna()

In [15]:
#scaling
def maximum_absolute_scaling(df):
    # copy the dataframe
    df_scaled = df.copy()
    # apply maximum absolute scaling
    for column in df_scaled.columns:
        df_scaled[column] = df_scaled[column]  / df_scaled[column].abs().max()
    return df_scaled

# call the maximum_absolute_scaling function
data = maximum_absolute_scaling(data)

In [16]:
#subsetting for only positive and non NA values
data = data.where(data>0)
data = data.dropna()
data.head()

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-11-22 21:00:00,0.312510,0.826160,0.675247,0.087633,0.133198,0.045455,0.050420,0.033333,0.012821,0.00625
2025-11-22 22:00:00,0.299997,0.798183,0.705316,0.086814,0.069028,0.033291,0.050420,0.044444,0.012821,0.00625
2025-11-22 23:00:00,0.291081,0.784075,0.735867,0.084767,0.055061,0.023047,0.033613,0.022222,0.012821,0.00625
2025-11-23 00:00:00,0.309422,0.823769,0.657686,0.088862,0.056883,0.021767,0.028011,0.011111,0.006410,0.00625
2025-11-23 01:00:00,0.351337,0.848637,0.596103,0.098280,0.068219,0.024968,0.033613,0.022222,0.012821,0.00625


In [17]:
data.to_csv(r'MOD-00691_timeseries_hourly_scaled.csv')

## Implementing NMF

In [18]:
#setting up the NMF
nmf = NMF(n_components = 4, max_iter = 8000)

In [19]:
W = nmf.fit_transform(data)
H = nmf.components_
V = pd.DataFrame(np.dot(W,H), columns=data.columns)
V.index = data.index
V

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-11-22 21:00:00,0.361654,0.806509,0.672804,0.092270,0.126502,0.045629,0.050380,0.035758,0.016212,0.009404
2025-11-22 22:00:00,0.344440,0.779262,0.703781,0.088989,0.077919,0.022123,0.036487,0.038714,0.026934,0.019872
2025-11-22 23:00:00,0.339105,0.764050,0.734001,0.086824,0.059625,0.011250,0.024165,0.026903,0.019037,0.013411
2025-11-23 00:00:00,0.353114,0.805396,0.656092,0.089874,0.062774,0.007970,0.017419,0.020173,0.013124,0.007884
2025-11-23 01:00:00,0.365878,0.842389,0.595923,0.093543,0.077233,0.013928,0.023174,0.026273,0.016836,0.010617
...,...,...,...,...,...,...,...,...,...,...
2025-12-31 14:00:00,0.397745,0.876550,0.789992,0.100575,0.140511,0.050671,0.052540,0.030804,0.009477,0.003120
2025-12-31 15:00:00,0.386672,0.855372,0.790425,0.098284,0.125536,0.044319,0.049745,0.033664,0.014466,0.007861
2025-12-31 16:00:00,0.393297,0.874151,0.729391,0.100422,0.149039,0.056533,0.060923,0.042035,0.018490,0.010772


In [20]:
W_df = pd.DataFrame(W, columns =[f'Feature {i+1}' for i in range(0,4)]) #array-like of shape (n_samples, n_components)
W_df

,Feature 1,Feature 2,Feature 3,Feature 4
0,0.095264,0.028270,0.155352,0.014708
1,0.090687,0.011366,0.169545,0.031080
2,0.087812,0.005218,0.182541,0.020975
3,0.100756,0.003924,0.145119,0.012331
4,0.110194,0.007429,0.116488,0.016605
...,...,...,...,...
889,0.100181,0.032465,0.192145,0.004879
890,0.096866,0.027614,0.194872,0.012295
891,0.102353,0.035174,0.169156,0.016847
892,0.109955,0.037795,0.127250,0.028021


In [21]:
COLS_TO_INCLUDE.pop(0)
COLS_TO_INCLUDE

['co', 'no', 'no2', 'o3', 'bin0', 'bin1', 'bin2', 'bin3', 'bin4', 'bin5']

In [22]:
H_df = pd.DataFrame(H, index = [f'Feature {i+1}' for i in range(0,4)], columns = COLS_TO_INCLUDE) #array-like of shape (n_components, n_features)
H_df

,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
Feature 1,2.704450,6.452602,2.351793,0.685420,0.516545,0.000000,0.017091,0.076883,0.038208,0.000000
Feature 2,0.735698,1.042616,0.000000,0.190348,2.734094,1.532162,1.298044,0.578712,0.064036,0.000000
Feature 3,0.535671,1.012898,2.883999,0.132046,0.000000,0.001144,0.033335,0.002197,0.000000,0.000000
Feature 4,0.000000,0.338297,0.049407,0.073345,0.000000,0.145253,0.467559,0.797685,0.731711,0.639377


In [23]:
#converting the results to a dataframe
results = pd.DataFrame(W,index=data.index) #H is time series data of the factors, W is the comp (coeff matrix)
results.columns = ["Factor {}".format(i+1) for i in range(H.T.shape[1])]
results

,Factor 1,Factor 2,Factor 3,Factor 4
timestamp_local,,,,
2025-11-22 21:00:00,0.095264,0.028270,0.155352,0.014708
2025-11-22 22:00:00,0.090687,0.011366,0.169545,0.031080
2025-11-22 23:00:00,0.087812,0.005218,0.182541,0.020975
2025-11-23 00:00:00,0.100756,0.003924,0.145119,0.012331
2025-11-23 01:00:00,0.110194,0.007429,0.116488,0.016605
...,...,...,...,...
2025-12-31 14:00:00,0.100181,0.032465,0.192145,0.004879
2025-12-31 15:00:00,0.096866,0.027614,0.194872,0.012295
2025-12-31 16:00:00,0.102353,0.035174,0.169156,0.016847


In [24]:
COLS_TO_INCLUDE = ['co','no','no2','o3','bin0','bin1','bin2','bin3','bin4','bin5']
comp = pd.DataFrame(H, index = results.columns, columns = COLS_TO_INCLUDE)
comp

,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
Factor 1,2.704450,6.452602,2.351793,0.685420,0.516545,0.000000,0.017091,0.076883,0.038208,0.000000
Factor 2,0.735698,1.042616,0.000000,0.190348,2.734094,1.532162,1.298044,0.578712,0.064036,0.000000
Factor 3,0.535671,1.012898,2.883999,0.132046,0.000000,0.001144,0.033335,0.002197,0.000000,0.000000
Factor 4,0.000000,0.338297,0.049407,0.073345,0.000000,0.145253,0.467559,0.797685,0.731711,0.639377


In [25]:
res = []

for col in comp.columns:
    #for each factor, compute its contribution to the species in V
    by_factor = pd.Series(dtype=float)
    for i, factor in enumerate(results.columns):
        contrib = H[i, data.columns.get_loc(col)] * W[:, i].sum()
        by_factor.at[factor] = contrib

    #normalizing by the total amount of that species in the original data
    by_factor /= data[col].sum()

    #adding as a row to the resulting dataframe
    res.append(pd.DataFrame(by_factor).T.rename(index={0: col}))

res = pd.concat(res)
res.columns = results.columns
res['Residual'] = 1 - res.sum(axis=1)

res

,Factor 1,Factor 2,Factor 3,Factor 4,Residual
co,0.689739,0.102636,0.207209,0.000000,0.000417
no,0.661910,0.100550,0.193407,0.038746,0.005386
no2,0.738112,0.065239,0.175735,0.021169,-0.000254
o3,0.348256,0.000000,0.647737,0.004002,0.000005
bin0,0.257171,0.744596,0.000000,0.000000,-0.001767
bin1,0.000000,0.933986,0.001934,0.088549,-0.024468
bin2,0.016594,0.689392,0.049089,0.248334,-0.003410
bin3,0.092128,0.379334,0.003992,0.522894,0.001652
bin4,0.080671,0.073957,0.000000,0.845125,0.000246
bin5,0.000000,0.000000,0.000000,1.099089,-0.099089


In [26]:
results.to_csv(r'MOD-00691_4_factor_results.csv')
comp.to_csv(r'MOD-00691_4_factor_comp.csv')
res.to_csv(r'MOD-00691_4_factor_resid.csv')